In [10]:
import pandas as pd
import numpy as np
import os
from glob import glob
import yaml
from tqdm import tqdm

In [5]:
yaml_files = glob(os.path.join("./raw_data", '*.yaml'))
yaml_files[0]

'./raw_data/598067.yaml'

## Deliveries Df

In [6]:
def get_phase(ball):
    if 0 <= ball < 6:
        return 'powerplay'
    elif 6 <= ball < 16:
        return 'middle'
    elif 16 <= ball < 20:
        return 'death'
    return 'unknown'

In [11]:
deliveries_data = []

for file_path in tqdm(yaml_files):
    match_id = os.path.splitext(os.path.basename(file_path))[0]

    with open(file_path, 'r') as file:
        match_data = yaml.safe_load(file)
    
    teams = match_data['info']['teams']
    opp_map = {
        teams[i]:  teams[1 - i]
        for i in range(2)
    }
    
    for innings_data_ in match_data['innings']:
        (innings, innings_data), = innings_data_.items()
        for ball_data_ in innings_data['deliveries']:
            (ball, ball_data), = ball_data_.items()
            deliveries_data.append({
                'match_id': match_id,
                'innings': innings,
                'ball': ball,
                'over': int(ball),
                'batting_team': innings_data['team'],
                'bowling_team': opp_map[innings_data['team']],
                'striker': ball_data['batsman'],
                'non_striker': ball_data['non_striker'],
                'bowler': ball_data['bowler'],
                'runs_off_bat': ball_data['runs']['batsman'],
                'extras': ball_data['runs']['extras'],
                'total_runs': ball_data['runs']['total'],
                'wicket_flag': int('wicket' in ball_data),
                'phase': get_phase(ball)
            })

deliveries_df = pd.DataFrame(deliveries_data)

100%|██████████| 1169/1169 [02:51<00:00,  6.80it/s]


In [12]:
deliveries_df.head()

,match_id,innings,ball,over,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,total_runs,wicket_flag,phase
0,598067,1st innings,0.1,0,Pune Warriors,Delhi Daredevils,RV Uthappa,AJ Finch,IK Pathan,0,0,0,0,powerplay
1,598067,1st innings,0.2,0,Pune Warriors,Delhi Daredevils,RV Uthappa,AJ Finch,IK Pathan,2,0,2,0,powerplay
2,598067,1st innings,0.3,0,Pune Warriors,Delhi Daredevils,RV Uthappa,AJ Finch,IK Pathan,4,0,4,0,powerplay
3,598067,1st innings,0.4,0,Pune Warriors,Delhi Daredevils,RV Uthappa,AJ Finch,IK Pathan,0,0,0,0,powerplay
4,598067,1st innings,0.5,0,Pune Warriors,Delhi Daredevils,RV Uthappa,AJ Finch,IK Pathan,1,0,1,0,powerplay


In [13]:
deliveries_df['phase'].unique()

array(['powerplay', 'middle', 'death'], dtype=object)

In [14]:
deliveries_df.to_csv("./data/deliveries.csv", index=False)

## Match Info Df

In [16]:
match_info = []

for file_path in tqdm(yaml_files):
    match_id = os.path.splitext(os.path.basename(file_path))[0]

    with open(file_path, 'r') as file:
        match_data = yaml.safe_load(file)
    
    teams = tuple(sorted(match_data['info']['teams']))
    
    if 'result' in match_data['info']['outcome']:
        outcome = 'no-result'
    elif 'winner' in match_data['info']['outcome']: 
        outcome = match_data['info']['outcome']['winner']
    else: # * Safety check
        print("Smthg at fault:", match_data['info']['outcome'])
    
    match_info.append({
        'date': match_data['info']['dates'][0],
        'date_raw': match_data['info']['dates'], # * general check
        'venue': match_data['info']['venue'],
        'teams': teams,
        'toss_winner': match_data['info']['toss']['winner'],
        'toss_decision': match_data['info']['toss']['decision'],
        'match_winner': outcome,
        'players': tuple(
            tuple(sorted(match_data['info']['players'][team]))
            for team in teams
        )
    })

match_info_df = pd.DataFrame(match_info)

100%|██████████| 1169/1169 [02:50<00:00,  6.87it/s]


In [17]:
match_info_df.head()

,date,date_raw,venue,teams,toss_winner,toss_decision,match_winner,players
0,2013-05-19,[2013-05-19],Subrata Roy Sahara Stadium,"(Delhi Daredevils, Pune Warriors)",Pune Warriors,bat,Pune Warriors,"((B Chipli, BJ Rohrer, CM Gautam, DA Warner, D..."
1,2015-04-15,[2015-04-15],Maharashtra Cricket Association Stadium,"(Delhi Daredevils, Kings XI Punjab)",Kings XI Punjab,bat,Delhi Daredevils,"((A Mishra, AD Mathews, DJ Muthuswami, Imran T..."
2,2011-05-14,[2011-05-14],M Chinnaswamy Stadium,"(Kolkata Knight Riders, Royal Challengers Bang...",Royal Challengers Bangalore,field,Royal Challengers Bangalore,"((B Lee, EJG Morgan, G Gambhir, Iqbal Abdulla,..."
3,2017-05-07,[2017-05-07],"Punjab Cricket Association IS Bindra Stadium, ...","(Gujarat Lions, Kings XI Punjab)",Gujarat Lions,field,Gujarat Lions,"((AJ Finch, Ankit Soni, Basil Thampi, DR Smith..."
4,2010-04-17,[2010-04-17],M Chinnaswamy Stadium,"(Mumbai Indians, Royal Challengers Bangalore)",Royal Challengers Bangalore,field,Mumbai Indians,"((AM Nayar, AP Tare, AT Rayudu, CRD Fernando, ..."


In [22]:
[it for it in match_info_df['date_raw'] if len(it) != 1]

[[datetime.date(2017, 5, 17), datetime.date(2017, 5, 18)],
 [datetime.date(2014, 5, 27), datetime.date(2014, 5, 28)]]

In [23]:
match_info_df.drop(columns=['date_raw']).to_csv("./data/matches_info.csv", index=False)